In [5]:
%pwd

'e:\\Health_Care_Assistant_ChatBot-End-to-End-RAG-Implementaion\\research'

In [6]:
import os
os.chdir("../")

In [7]:
%pwd

'e:\\Health_Care_Assistant_ChatBot-End-to-End-RAG-Implementaion'

In [8]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat, ConversionStatus
from docling.datamodel.pipeline_options import PdfPipelineOptions

from pathlib import Path

from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

e:\Health_Care_Assistant_ChatBot-End-to-End-RAG-Implementaion\medibot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
## joing the .md files from the Data/Output folder

import os

# Directory where your markdown files are saved (from the previous script)
output_dir = Path("Data/Output")

# The name of the final merged file
merged_file = Path("Data/Output/medical_data.md")

# Find all markdown files in the directory
md_files = list(output_dir.glob("*.md"))

print(f"Found {len(md_files)} Markdown files to merge.")

# Open the merged file in write mode
with open(merged_file, "w", encoding="utf-8") as outfile:
    for md_file in md_files:
        print(f"Merging: {md_file.name}...")
        
        # Read each individual markdown file
        with open(md_file, "r", encoding="utf-8") as infile:
            # (Optional) Add a header to indicate which file the following text came from
            outfile.write(f"\n\n# --- Source: {md_file.name} ---\n\n")
            
            # Write the content to the merged file
            outfile.write(infile.read())
            
            # Add some spacing between files
            outfile.write("\n\n")
            
print(f"\n✅ Successfully merged all files into {merged_file.name}")


Found 2 Markdown files to merge.
Merging: Davidson's Principles & Practice of Medicine-pages-1.md...
Merging: Davidson's Principles & Practice of Medicine-pages-2.md...

✅ Successfully merged all files into medical_data.md


In [ ]:
### Filtering out the meta data

from typing import  List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of document objects, return a new list of document objects containing only
    'source' in metadata and the original page_content.
    """

    minimal_docs: List[Document] = []

    for doc in docs:
        src = doc.metadata.get('source')
        minimal_docs.append(
            Document(
                page_content = doc.page_content,
                metadata= {'source':src}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs


In [ ]:
## split the documents into smaller chunks

def text_split(minimal_docs):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 20,
    )

    text_chunk = text_splitter.split_documents(minimal_docs)
    return text_chunk
    

In [ ]:
text_chunk = text_split(minimal_docs)
print(f"Number of chunks :{len(text_chunk)}")

In [ ]:

# Preview chunks
print(f"\n--- Chunk Examples ---")
for i, chunk in enumerate(text_chunk[34:36]):
    print(f"\nChunk {i+1} (length: {len(chunk.page_content)} chars):")
    print(f"{chunk.page_content[:200]}...")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN not found in environment variables")

os.environ["HF_TOKEN"] = hf_token

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
def download_embeddings():
    """
    Download and return the HuggingFace embeddings model
    """

    embeddings = HuggingFaceEmbeddings(
        model_name = "sentence-transformers/all-MiniLM-L6-v2",
        
    )
    return embeddings

embedding = download_embeddings()

In [ ]:


# Create ChromaDB vector store
print(f"Creating ChromaDB vector store from {len(text_chunk)} chunks...")
print("This may take a minute...\n")

# Set persistent directory
persist_directory = "./chroma_db"

# Create vector store
vectorstore = Chroma.from_documents(
    documents=text_chunk,
    embedding=embedding,
    persist_directory=persist_directory,
    collection_name="local_rag_collection"
)

print(f"✓ ChromaDB vector store created successfully!")
print(f"✓ Indexed {len(text_chunk)} document chunks")
print(f"✓ Stored at: {persist_directory}")
print(f"\nℹ️  Vector store persisted to disk - you can reload it later!")

In [ ]:
# Create retriever

retriever = vectorstore.as_retriever(
    search_type="similarity",    # Use cosine similarity
    search_kwargs={"k": 4}        # Retrieve top 4 most relevant chunks
)

print("✓ Retriever configured successfully")
print(f"  - Search type: similarity")
print(f"  - Number of documents to retrieve (k): 4")

# Test retrieval
test_query = "What are the symptoms of diabetes?"
print(f"\n--- Retriever Test ---")
print(f"Query: '{test_query}'")

retrieved_docs = retriever.invoke(test_query)

print(f"\nRetrieved {len(retrieved_docs)} documents:")
for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print(f"  Content preview: {doc.page_content[:150]}...")
    print(f"  Source: Page {doc.metadata.get('page', 'N/A')}")



In [ ]:
retrieved_docs[0].page_content[:500]


In [ ]:
# checking the llm connection
llm = ChatOllama(
    model="gemma:2b",
    temperature=0,          # Deterministic responses (0 = focused, 1 = creative)
    )

print("✓ LLM configured successfully")
print(f"  - Model: gemma3:1b (local)")
print(f"  - Temperature: 0 (deterministic)")

# Test LLM
test_response = llm.invoke("Say 'Hello! I am Gemma running locally!'")
print(f"\nLLM Test Response: {test_response.content}")

In [ ]:
# Define prompt template
system_prompt = (

    '''
    You are a medical assistant chatbot powered by Retrieval-Augmented Generation (RAG).

Your primary objective is to provide accurate, safe, and evidence-based medical information strictly grounded in the retrieved context.

-----------------------------------
CORE BEHAVIOR
-----------------------------------
- ONLY use the provided context to generate answers.
- If the answer is not clearly supported by the context, say:
  "I don't have enough information in the provided sources to answer that."
- Do NOT hallucinate, infer, or fabricate medical facts.

-----------------------------------
SAFETY & MEDICAL DISCLAIMER
-----------------------------------
- You are NOT a licensed medical professional.
- Always include a brief disclaimer when giving health-related guidance:
  "This information is for educational purposes only and not a substitute for professional medical advice."
- If the query involves severe symptoms, emergencies, or life-threatening conditions:
  → Strongly recommend seeking immediate medical attention.

-----------------------------------
RESPONSE STYLE
-----------------------------------
- Be clear, concise, and structured.
- Use simple language unless technical detail is necessary.
- Prefer bullet points for symptoms, causes, and treatments.
- Avoid unnecessary verbosity.

-----------------------------------
CONTEXT USAGE (RAG)
-----------------------------------
- Base your response strictly on the retrieved documents.
- If multiple sources conflict:
  → Acknowledge uncertainty and present both perspectives.
- Prioritize:
  1. Clinical guidelines
  2. Peer-reviewed sources
  3. Trusted medical institutions

-----------------------------------
QUERY HANDLING
-----------------------------------
1. If the query is:
   - Symptom-based → Provide possible causes (not diagnosis)
   - Drug-related → Include usage, side effects, and warnings
   - Disease-related → Include overview, symptoms, causes, treatment

2. If the query is vague:
   → Ask clarifying questions before answering.

-----------------------------------
RESTRICTIONS
-----------------------------------
- Do NOT provide:
  - Exact diagnoses
  - Prescription instructions
  - Dosage recommendations beyond general info in context
- Do NOT speculate beyond retrieved data.

-----------------------------------
TONE
-----------------------------------
- Professional, calm, and non-alarming
- Supportive but not overly emotional
- Avoid definitive or absolute claims

-----------------------------------
FAIL-SAFE
-----------------------------------
If context is missing or insufficient:
→ Respond with:
"I’m not confident in providing an accurate answer based on the available information. Please consult a healthcare professional."

-----------------------------------
OUTPUT FORMAT
-----------------------------------
- Start with a direct answer
- Follow with supporting details
- End with disclaimer (if medical)

-----------------------------------
END OF INSTRUCTIONS
-----------------------------------'''
    
    "Context: {context}\n\n"
    "query: {question}"
)

prompt = ChatPromptTemplate.from_template(system_prompt)

# Helper function to format documents
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

# Build RAG chain using LCEL
rag_chain = (
    {
        "context": retriever | format_docs,  # Retrieve and format docs
        "question": RunnablePassthrough()      # Pass through the question
    }
    | prompt           # Format with prompt template
    | llm              # Generate answer with local LLM
    | StrOutputParser() # Parse output to string
)

print("✓ RAG chain created successfully using LCEL!")
print("\nRAG Pipeline Flow:")
print("  1. User provides a query")
print("  2. Retriever finds top 4 relevant chunks (local ChromaDB)")
print("  3. Chunks are formatted as context")
print("  4. Context + question formatted with prompt template")
print("  5. Local LLM (gemma3:1b) generates answer")
print("  6. Answer parsed and returned")


In [ ]:
# Example Query 1: General question
query1 = "What are the common symptoms of diabetes?"

print(f"Query: {query1}")
print("\nProcessing locally...\n")

answer = rag_chain.invoke(query1)

print("=" * 80)
print("ANSWER:")
print("=" * 80)
print(answer)
print("\n" + "=" * 80)

# Show source documents
print("\nSOURCE DOCUMENTS USED:")
print("=" * 80)
retrieved_docs = retriever.invoke(query1)
for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print(f"  Page: {doc.metadata.get('page', 'N/A')}")
    print(f"  Content: {doc.page_content[:200]}...")
    print("-" * 80)

In [ ]:
# Interactive Q&A
def ask_question(question):
    """Ask a question to the RAG system."""
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}")
    
    answer = rag_chain.invoke(question)
    
    print(f"\nAnswer: {answer}")
    print(f"{'='*80}\n")
    
    return answer

my_question = "Wha t are the treatment options for DIABETES?"
ask_question(my_question)